In [1]:
# Testando os benckmarks para cada dataset (cada especiaria) com cada target (Import, Export, Production)


import os
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

dataframes = {}
datasets_path = "datasets"
targets = ["Import", "Export", "Production"]

for filename in os.listdir(datasets_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(datasets_path, filename)
        df = pd.read_csv(file_path)

        df = df[(df[f"is_outlier_Import"] == False) & (df[f"is_outlier_Export"] == False) & (df[f"is_outlier_Production"] == False)]
        df = df[["Area", "Year", "Item", "Import", "Export", "Production"]]

        # Salva o dataframe tratado como CSV na pasta "datasets_tratados"
        output_dir = "datasets_tratados"
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, filename)
        df.to_csv(output_path, index=False)

        for item in targets:
            print(f"Benchmarking da especiaria {filename} para {item}")

            # Retirei os outliers


            X = df.drop(columns=[item])
            X = pd.get_dummies(X, columns=["Area", "Item"])
            y = df[item]


            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            modelos = {
                "Regressão Linear": LinearRegression(),
                "Árvore de Decisão": DecisionTreeRegressor(),
                "Random Forest": RandomForestRegressor(),
                "SVM": SVR(),
                #"Rede Neural (MLP)": MLPRegressor(max_iter=1000)
            }

            for nome, modelo in modelos.items():
                modelo.fit(X_train, y_train)
                y_pred = modelo.predict(X_test)

                with open("resultados_benchmark.txt", "a", encoding="utf-8") as f:
                    f.write(f"Especiaria: {filename} para {item}\n")
                    f.write(f"🔹 {nome}\n")
                    f.write(f"MAE: {mean_absolute_error(y_test, y_pred)}\n")
                    f.write(f"MSE: {mean_squared_error(y_test, y_pred)}\n")
                    f.write(f"R²: {r2_score(y_test, y_pred)}\n")
                    f.write("-" * 40 + "\n")

            







Benchmarking da especiaria dados_Cinnamon and cinnamon-tree flowers, raw.csv para Import
Benchmarking da especiaria dados_Cinnamon and cinnamon-tree flowers, raw.csv para Export
Benchmarking da especiaria dados_Cinnamon and cinnamon-tree flowers, raw.csv para Production
Benchmarking da especiaria dados_Ginger, raw.csv para Import
Benchmarking da especiaria dados_Ginger, raw.csv para Export
Benchmarking da especiaria dados_Ginger, raw.csv para Production
Benchmarking da especiaria dados_Chillies and peppers, dry (Capsicum spp., Pimenta spp.), raw.csv para Import
Benchmarking da especiaria dados_Chillies and peppers, dry (Capsicum spp., Pimenta spp.), raw.csv para Export
Benchmarking da especiaria dados_Chillies and peppers, dry (Capsicum spp., Pimenta spp.), raw.csv para Production
Benchmarking da especiaria dados_Vanilla, raw.csv para Import
Benchmarking da especiaria dados_Vanilla, raw.csv para Export
Benchmarking da especiaria dados_Vanilla, raw.csv para Production
Benchmarking da es

# Relatório de Destaque dos Algoritmos por Especiaria e Tipo (Baseado no Maior R²)

A tabela abaixo apresenta o algoritmo que mais se destacou (maior R²) para cada combinação de especiaria e tipo de previsão (Importação, Exportação, Produção), com base nos resultados do benchmark.

| Especiaria | Importação | Exportação | Produção |
|---|---|---|---|
| Canela e flores de caneleira | Random Forest | Árvore de Decisão | Árvore de Decisão |
| Gengibre | Random Forest | Árvore de Decisão | Random Forest |
| Pimentas secas (Capsicum/Pimenta) | Random Forest | Árvore de Decisão | Random Forest |
| Baunilha | Random Forest | Random Forest | Árvore de Decisão |
| Cravo (talos inteiros) | Random Forest | Random Forest | Random Forest |
| Pimenta (Piper spp.) | Random Forest | Random Forest | Random Forest |
| Noz-moscada, macis e cardamomos | Random Forest | Random Forest | Random Forest |
| Pimentas verdes (Capsicum/Pimenta) | Random Forest | Random Forest | Random Forest |
| Anis, badiana, coentro, cominho, alcaravia, funcho e zimbro | Árvore de Decisão | Árvore de Decisão | Árvore de Decisão |

**Critério:** O algoritmo destacado é o que apresentou o maior valor de R² (coeficiente de determinação) para cada caso.

**Observações:**
- O Random Forest foi o algoritmo mais consistente, apresentando o melhor desempenho na maioria das combinações.
- Em alguns casos, a Árvore de Decisão se destacou, especialmente em Exportação e Produção para Canela, e em todas as previsões para o grupo de Anis, Badiana, etc.
- SVM apresentou desempenho inferior em todos os cenários.


In [2]:
import os
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

dataframes = {}
datasets_path = "datasets_tratados"

for filename in os.listdir(datasets_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(datasets_path, filename)
        df = pd.read_csv(file_path)

        print(f"\n### Treinando para {filename} ###")

        # === Features e target ===
        X = df.drop(columns=["Import"])
        X = pd.get_dummies(X, columns=["Area", "Item"])
        y = df["Import"]

        # === Split: treino (80%) / teste (20%) ===
        X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # === Padronização ===
        scaler = StandardScaler()
        X_train_val_scaled = scaler.fit_transform(X_train_val)
        X_test_scaled = scaler.transform(X_test)

        # === Definição do modelo e grade de hiperparâmetros ===
        model = RandomForestRegressor(random_state=42)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5]
        }

        # === K-Fold Cross Validation ===
        kfold = KFold(n_splits=5, shuffle=True, random_state=42)
        grid_search = GridSearchCV(
            model,
            param_grid,
            cv=kfold,
            scoring='r2',
            n_jobs=-1
        )

        grid_search.fit(X_train_val_scaled, y_train_val)

        # === Melhor modelo ===
        best_model = grid_search.best_estimator_

        # === Avaliação final no conjunto de teste ===
        y_pred = best_model.predict(X_test_scaled)

        print("Melhores hiperparâmetros:", grid_search.best_params_)
        print("R² no teste:", r2_score(y_test, y_pred))
        print("MAE:", mean_absolute_error(y_test, y_pred))
        print("MSE:", mean_squared_error(y_test, y_pred))



### Treinando para dados_Cinnamon and cinnamon-tree flowers, raw.csv ###
Melhores hiperparâmetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
R² no teste: 0.9448106453800744
MAE: 182.68923456989253
MSE: 437570.06198830606

### Treinando para dados_Ginger, raw.csv ###
Melhores hiperparâmetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
R² no teste: 0.9592683272601221
MAE: 570.5602137430168
MSE: 4759948.716308412

### Treinando para dados_Chillies and peppers, dry (Capsicum spp., Pimenta spp.), raw.csv ###
Melhores hiperparâmetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
R² no teste: 0.9486404806066241
MAE: 625.3008235789475
MSE: 6851518.459208093

### Treinando para dados_Vanilla, raw.csv ###
Melhores hiperparâmetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
R² no teste: 0.9277704828251055
MAE: 10.416954502369663
MSE: 1594.7418538972072

### Treinando para dados_Cloves (whole stems), raw.csv ##

# Comparativo de R² Antes e Pós-Refinamento (Importação)

A tabela abaixo compara o melhor R² obtido antes e após o ajuste de hiperparâmetros para cada especiaria (Importação), mostrando a evolução do desempenho dos modelos.

| Especiaria                                               | R² Antes (Melhor Algoritmo) | R² Pós-Refino | Evolução   |
|----------------------------------------------------------|-----------------------------|--------------|------------|
| Canela e flores de caneleira                            | 0.9413                      | 0.9448       | ▲ +0.0035  |
| Gengibre                                                | 0.9573                      | 0.9593       | ▲ +0.0020  |
| Pimentas secas (Capsicum/Pimenta)                       | 0.9494                      | 0.9486       | ▼ -0.0008  |
| Baunilha                                                | 0.9304                      | 0.9278       | ▼ -0.0026  |
| Cravo (talos inteiros)                                  | 0.7381                      | 0.7370       | ▼ -0.0011  |
| Pimenta (Piper spp.)                                    | 0.9810                      | 0.9803       | ▼ -0.0007  |
| Noz-moscada, macis e cardamomos                         | 0.8771                      | 0.9086       | ▲ +0.0315  |
| Pimentas verdes (Capsicum/Pimenta)                      | 0.9858                      | 0.9843       | ▼ -0.0015  |
| Anis, badiana, coentro, cominho, alcaravia, funcho, etc | 0.9094                      | 0.8617       | ▼ -0.0477  |

**Legenda:**  
▲: Melhora no R²  
▼: Queda no R²

**Observações:**
- Houve leve melhora no R² para Canela, Gengibre e Noz-moscada, macis e cardamomos, sendo este último o que mais evoluiu (+0.0315).
- Para a maioria das especiarias, o R² manteve-se estável, com variações pequenas.
- Em alguns casos, houve pequena redução no R² após o ajuste de hiperparâmetros.